## TF-IDF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

In [ ]:
df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')

In [ ]:
len(df)

In [ ]:
df.head(10)

In [ ]:
# df["label"] = df["label"].astype(int)

In [ ]:
df_neural = pd.read_csv("/content/myneural.csv", encoding="utf-8-sig")

In [ ]:
df_human = pd.read_csv("/content/myhuman.csv", encoding="utf-8-sig")

In [ ]:
df_train_val, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)

In [ ]:
print(f"Всего: {len(df)}, Тестовая: {len(df_test)}, Остаток (train+val): {len(df_train_val)}")

In [ ]:
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.10,
    stratify=df_train_val['label'],
    random_state=42
)

In [ ]:
print(f"Обучающая: {len(df_train)}, Валидационная: {len(df_val)}")

In [ ]:
df_train.to_csv('/content/train.csv', index=False, encoding='utf-8-sig')
df_val.to_csv  ('/content/val.csv',   index=False, encoding='utf-8-sig')
df_test.to_csv ('/content/test.csv',  index=False, encoding='utf-8-sig')

In [ ]:
X_train = df_train['clean_text'].str.lower().values
X_val   = df_val  ['clean_text'].str.lower().values
X_test  = df_test ['clean_text'].str.lower().values

y_train = df_train['label'].values
y_val   = df_val  ['label'].values
y_test  = df_test ['label'].values

### **TF-IDF**

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)

In [ ]:
X_val_tfidf  = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
print('Train TF-IDF shape:', X_train_tfidf.shape)
print('Val   TF-IDF shape:', X_val_tfidf.shape)
print('Test  TF-IDF shape:', X_test_tfidf.shape)

In [ ]:
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

## XGBoost

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb
import numpy as np

In [ ]:
df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')
df_train_val, df_test = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
df_train,     df_val  = train_test_split(df_train_val, test_size=0.10, stratify=df_train_val['label'], random_state=42)

X_train_text = df_train['clean_text'].str.lower().values
X_val_text   = df_val  ['clean_text'].str.lower().values
X_test_text  = df_test ['clean_text'].str.lower().values

y_train = df_train['label'].values
y_val   = df_val  ['label'].values
y_test  = df_test ['label'].values

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=5000,        # ограничиваем словарь до 5000 самых частотных терминов
    stop_words='english',
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf   = tfidf.transform(X_val_text)
X_test_tfidf  = tfidf.transform(X_test_text)

In [ ]:
# Вариант: конвертация в плотный массив (если хватит памяти)
# X_train_dense = X_train_tfidf.toarray()
# X_val_dense   = X_val_tfidf.toarray()
# X_test_dense  = X_test_tfidf.toarray()

In [ ]:
# Вариант: понижение размерности с помощью TruncatedSVD до 500 компонент
svd = TruncatedSVD(n_components=500, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd   = svd.transform(X_val_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)

In [ ]:
dtrain = xgb.DMatrix(X_train_svd, label=y_train)
dval   = xgb.DMatrix(X_val_svd,   label=y_val)
dtest  = xgb.DMatrix(X_test_svd)

In [ ]:
neg, pos = np.bincount(y_train)
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'scale_pos_weight': neg / pos,
    'seed': 42
}

In [ ]:
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dval, 'validation')],
    early_stopping_rounds=10,
    verbose_eval=10
)

In [ ]:
y_prob = bst.predict(dtest)
y_pred = (y_prob >= 0.5).astype(int)

In [ ]:
from xgboost import XGBClassifier
import numpy as np

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

In [ ]:
clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

### Обучение

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
clf.fit(X_train_tfidf, y_train)

In [ ]:
import joblib
joblib.dump(clf, '/content/xgb_clf.pkl')

### Валидация

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
y_val_pred = clf.predict(X_val_tfidf)

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
print("\nClassification Report (валидация):")
print(classification_report(y_val, y_val_pred, target_names=['human', 'AI']))

### Тест

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

In [ ]:
y_pred = clf.predict(X_test_tfidf)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
print("\nClassification Report (тест):")
print(classification_report(y_test, y_pred, target_names=['human', 'AI']))

In [ ]:
prec  = precision_score(y_test, y_pred, pos_label=1)
rec   = recall_score(y_test, y_pred, pos_label=1)
f1    = f1_score(y_test, y_pred, pos_label=1)
print(f"\nAI class — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve
)

In [ ]:
def plot_conf_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Human", "AI"])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(title)
    plt.show()

def plot_metrics_set(X, y_true, name, model):
    y_pred = model.predict(X)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    print(f"{name} — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
    plot_conf_matrix(y_true, y_pred, f"{name} — Confusion Matrix")

In [ ]:
# Графики по наборам
plot_metrics_set(X_train_tfidf, y_train, "Train", clf)
plot_metrics_set(X_val_tfidf,   y_val,   "Validation", clf)
plot_metrics_set(X_test_tfidf,  y_test,  "Test", clf)

In [ ]:
# ROC-кривая (тест)
y_probs_test = clf.predict_proba(X_test_tfidf)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_probs_test)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC-кривая (тест)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# PR-кривая (тест)
precision, recall, _ = precision_recall_curve(y_test, y_probs_test)

plt.figure()
plt.plot(recall, precision, label='PR curve')
plt.title('Precision-Recall кривая (тест)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# 1. Загрузим данные и приведём метки 0→'human', 1→'AI'
df = pd.read_csv('/content/mydata.csv', encoding='utf-8-sig')
label_mapping = {0: 'human', 1: 'AI'}
df['label'] = df['label'].map(label_mapping)

# (Доп. данные, если нужны)
# df_neural = pd.read_csv("/content/myneural.csv", encoding="utf-8-sig")
# df_human  = pd.read_csv("/content/myhuman.csv", encoding="utf-8-sig")

In [ ]:
# 2. Разбиваем на train+val и test
df_train_val, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)
print(f"Всего: {len(df)}, Тестовая: {len(df_test)}, Остаток (train+val): {len(df_train_val)}")

# 3. Разбиваем train+val на train и val
df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.10,
    stratify=df_train_val['label'],
    random_state=42
)
print(f"Обучающая: {len(df_train)}, Валидационная: {len(df_val)}")

# Сохраним CSV (если нужно)
df_train.to_csv('/content/train.csv', index=False, encoding='utf-8-sig')
df_val.to_csv(  '/content/val.csv',   index=False, encoding='utf-8-sig')
df_test.to_csv( '/content/test.csv',  index=False, encoding='utf-8-sig')

# 4. Подготовим X и y (нормализуем текст в нижний регистр)
X_train = df_train['clean_text'].str.lower().values
X_val   = df_val[  'clean_text'].str.lower().values
X_test  = df_test[ 'clean_text'].str.lower().values

y_train = df_train['label'].values
y_val   = df_val[  'label'].values
y_test  = df_test[ 'label'].values

In [ ]:
# 5. TF-IDF векторизация
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000,
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

print('Train TF-IDF shape:', X_train_tfidf.shape)
print('Val   TF-IDF shape:', X_val_tfidf.shape)
print('Test  TF-IDF shape:', X_test_tfidf.shape)

# Сохраним векторизатор
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

In [ ]:
# 6. Обучим логистическую регрессию с balanced-классами
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)
model.fit(X_train_tfidf, y_train)

# Сохраним модель
joblib.dump(model, '/content/logreg_balanced.pkl')

# 7. Валидация (без GridSearch) – классификация и матрица ошибок
y_val_pred = model.predict(X_val_tfidf)
print("=== Classification Report (валидация) ===")
print(classification_report(y_val, y_val_pred, target_names=['human', 'AI']))

print("=== Confusion Matrix (валидация) ===")
cm_val = confusion_matrix(y_val, y_val_pred, labels=['human', 'AI'])
disp_val = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=['human', 'AI']
)
disp_val.plot(cmap='Blues', values_format='d')
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок (Validation)')
plt.show()

# 8. Тестирование – классификация и матрица ошибок
y_test_pred = model.predict(X_test_tfidf)

cm = confusion_matrix(y_test, y_test_pred, labels=['human', 'AI'])
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix (Test):")
print(cm)
print(f"True Negatives (human→human): {tn}")
print(f"False Positives (human→AI):   {fp}")
print(f"False Negatives (AI→human):    {fn}")
print(f"True Positives (AI→AI):        {tp}\n")

print("Classification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=['human', 'AI']))

# Метрики для класса “AI”
prec  = precision_score(y_test, y_test_pred, pos_label='AI')
rec   = recall_score(   y_test, y_test_pred, pos_label='AI')
f1    = f1_score(       y_test, y_test_pred, pos_label='AI')
print(f"AI class — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}\n")

# Эксперимент с порогом probabilities
probs = model.predict_proba(X_test_tfidf)[:, 1]  # вероятности для "AI"
threshold = 0.6
y_test_thresh = np.where(probs >= threshold, 'AI', 'human')
prec_t = precision_score(y_test, y_test_thresh, pos_label='AI')
rec_t  = recall_score(   y_test, y_test_thresh, pos_label='AI')
f1_t   = f1_score(       y_test, y_test_thresh, pos_label='AI')
print(f"Threshold {threshold} — Precision: {prec_t:.3f}, Recall: {rec_t:.3f}, F1: {f1_t:.3f}\n")

# 9. Функции для визуализации метрик и матриц с нужными подписями
def plot_conf_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=['human', 'AI'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['human', 'AI']
    )
    disp.plot(cmap='Blues', values_format='d')
    plt.xlabel('Предсказанный класс')
    plt.ylabel('Истинный класс')
    plt.title(title)
    plt.show()

def plot_metrics_set(X, y_true, name, model):
    y_pred_local = model.predict(X)
    prec_local  = precision_score(y_true, y_pred_local, pos_label='AI')
    rec_local   = recall_score(y_true, y_pred_local, pos_label='AI')
    f1_local    = f1_score(y_true, y_pred_local, pos_label='AI')
    print(f"{name} — Precision: {prec_local:.3f}, Recall: {rec_local:.3f}, F1: {f1_local:.3f}")
    plot_conf_matrix(y_true, y_pred_local, f"{name} — Матрица ошибок")

# Визуализация метрик на train/val/test
plot_metrics_set(X_train_tfidf, y_train, "Train",      model)
plot_metrics_set(X_val_tfidf,   y_val,   "Validation", model)
plot_metrics_set(X_test_tfidf,  y_test,  "Test",       model)

# 10. ROC-кривая (тест)
y_test_bin = np.array([1 if lbl == 'AI' else 0 for lbl in y_test])
fpr, tpr, _ = roc_curve(y_test_bin, probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC-кривая (тест)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()

# 11. Precision-Recall кривая (тест)
precision_vals, recall_vals, _ = precision_recall_curve(y_test_bin, probs)

plt.figure()
plt.plot(recall_vals, precision_vals, label='PR curve')
plt.title('Precision-Recall кривая (тест)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

# 12. Визуализация TF-IDF → SVD → t-SNE (для всех данных)
X_all_tfidf = vectorizer.transform(df['clean_text'])

svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_all_tfidf)

tsne = TSNE(
    n_components=2,
    random_state=42,
    init='pca',
    learning_rate='auto'
)
X_tsne = tsne.fit_transform(X_svd)

plt.figure(figsize=(8, 6))
for label in ['human', 'AI']:
    idx = (df['label'] == label)
    plt.scatter(
        X_tsne[idx, 0],
        X_tsne[idx, 1],
        label=label,
        alpha=0.6,
        s=20
    )

plt.xlabel('t-SNE компонента 1')
plt.ylabel('t-SNE компонента 2')
plt.title('t-SNE проекция TF-IDF признаков (Logistic Regression)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# 13. XGBoost

# Переподготовим данные: снова разделим на train/val/test и TF-IDF + SVD
df_train_val, df_test = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
df_train,     df_val  = train_test_split(df_train_val, test_size=0.10, stratify=df_train_val['label'], random_state=42)

X_train_text = df_train['clean_text'].str.lower().values
X_val_text   = df_val['clean_text'].str.lower().values
X_test_text  = df_test['clean_text'].str.lower().values

y_train = df_train['label'].map({'human': 0, 'AI': 1}).values
y_val   = df_val['label'].map({'human': 0, 'AI': 1}).values
y_test  = df_test['label'].map({'human': 0, 'AI': 1}).values

tfidf_xgb = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=5000,
    stop_words='english',
    strip_accents='unicode',
    token_pattern=r'(?u)\b\w+\b'
)

X_train_tfidf_xgb = tfidf_xgb.fit_transform(X_train_text)
X_val_tfidf_xgb   = tfidf_xgb.transform(X_val_text)
X_test_tfidf_xgb  = tfidf_xgb.transform(X_test_text)

svd = TruncatedSVD(n_components=500, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf_xgb)
X_val_svd   = svd.transform(X_val_tfidf_xgb)
X_test_svd  = svd.transform(X_test_tfidf_xgb)

dtrain = xgb.DMatrix(X_train_svd, label=y_train)
dval   = xgb.DMatrix(X_val_svd,   label=y_val)
dtest  = xgb.DMatrix(X_test_svd)

neg, pos = np.bincount(y_train)
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'scale_pos_weight': neg / pos,
    'seed': 42
}

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dval, 'validation')],
    early_stopping_rounds=10,
    verbose_eval=10
)

y_prob = bst.predict(dtest)
y_pred = (y_prob >= 0.5).astype(int)

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
clf.fit(X_train_tfidf_xgb, y_train)
joblib.dump(clf, '/content/xgb_clf.pkl')

# Валидация XGBoost
y_val_pred = clf.predict(X_val_tfidf_xgb)
cm = confusion_matrix(y_val, y_val_pred)
print("Confusion Matrix (Validation XGBoost):\n", cm)
print("\nClassification Report (валидация XGBoost):")
print(classification_report(y_val, y_val_pred, target_names=['human', 'AI']))

# Тест XGBoost
y_test_pred = clf.predict(X_test_tfidf_xgb)
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix (Test XGBoost):\n", cm)
print("\nClassification Report (тест XGBoost):")
print(classification_report(y_test, y_test_pred, target_names=['human', 'AI']))

prec  = precision_score(y_test, y_test_pred, pos_label=1)
rec   = recall_score(y_test, y_test_pred, pos_label=1)
f1    = f1_score(y_test, y_test_pred, pos_label=1)
print(f"\nAI class — Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")

# Визуализация метрик XGBoost
def plot_conf_matrix_xgb(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['human', 'AI']
    )
    disp.plot(cmap='Blues', values_format='d')
    plt.xlabel('Предсказанный класс')
    plt.ylabel('Истинный класс')
    plt.title(title)
    plt.show()

def plot_metrics_set_xgb(X, y_true, name, model):
    y_pred_local = model.predict(X)
    prec_local  = precision_score(y_true, y_pred_local)
    rec_local   = recall_score(y_true, y_pred_local)
    f1_local    = f1_score(y_true, y_pred_local)
    print(f"{name} XGBoost — Precision: {prec_local:.3f}, Recall: {rec_local:.3f}, F1: {f1_local:.3f}")
    plot_conf_matrix_xgb(y_true, y_pred_local, f"{name} Матрица ошибок (XGBoost)")

plot_metrics_set_xgb(X_train_tfidf_xgb, y_train, "Train",      clf)
plot_metrics_set_xgb(X_val_tfidf_xgb,   y_val,   "Validation", clf)
plot_metrics_set_xgb(X_test_tfidf_xgb,  y_test,  "Test",       clf)

# ROC-кривая XGBoost (тест)
y_probs_test = clf.predict_proba(X_test_tfidf_xgb)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_probs_test)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC-кривая (XGBoost)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()

# PR-кривая XGBoost (тест)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_probs_test)

plt.figure()
plt.plot(recall_vals, precision_vals, label='PR curve')
plt.title('Precision-Recall кривая XGBoost (тест)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()
